[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YarinGaida/tabular-data-benchmarks/blob/main/supervised/supervised-deep.ipynb)

## Overview
In the `supervised-classic` notebook, we established strong baselines using traditional machine learning paradigms. We observed that while XGBoost dominates standard tabular data, high-dimensional biological data ($P \gg N$) requires extreme feature selection (like LASSO) to prevent the "Curse of Dimensionality."

In this notebook, we move to the cutting edge: **Deep Tabular Learning**. We will evaluate whether neural networks can overcome the lack of spatial inductive bias in tabular data and how they handle extreme dimensionality.

We will systematically evaluate these architectures across the exact same synthetic domains:
* **Standard Tabular Data:** A classic dataset typical of finance or standard clinical records ($N=1000, P=20$).
* **Biological Data ($P \gg N$):** A high-dimensional oncology dataset characterized by having significantly more features than samples ($N=500, P=2000$).

### Data Generation
We begin by generating our synthetic datasets, exactly as we did for the classical baselines, to ensure a fair comparison.

In [30]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
from pytorch_tabular import TabularModel
from pytorch_tabular.config import DataConfig, OptimizerConfig, TrainerConfig
from pytorch_tabular.models import CategoryEmbeddingModelConfig, FTTransformerConfig
from tabpfn import TabPFNClassifier
import logging
import warnings

# Ignore standard warnings and silence excessive library logs (keeping only the essential Seed)
warnings.filterwarnings('ignore')
logging.getLogger("pytorch_tabular").setLevel(logging.WARNING)
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)

plt.style.use('ggplot')

# Set device for GPU acceleration if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 1. Standard Tabular Data (N=1000, P=20)
X_std, y_std = make_classification(n_samples=1000, n_features=20, n_informative=5, random_state=42)
X_train_std, X_test_std, y_train_std, y_test_std = train_test_split(X_std, y_std, test_size=0.3, random_state=42)

# Convert to Pandas DataFrames (Required for Pytorch Tabular)
train_df_std = pd.DataFrame(X_train_std, columns=[f'feature_{i}' for i in range(X_train_std.shape[1])])
train_df_std['target'] = y_train_std
test_df_std = pd.DataFrame(X_test_std, columns=[f'feature_{i}' for i in range(X_test_std.shape[1])])

# 2. High-Dimensional Biological Data (N=500, P=2000)
X_bio, y_bio = make_classification(n_samples=500, n_features=1000, n_informative=10, random_state=42)
X_train_bio, X_test_bio, y_train_bio, y_test_bio = train_test_split(X_bio, y_bio, test_size=0.3, random_state=42)

# Convert to Pandas DataFrames
train_df_bio = pd.DataFrame(X_train_bio, columns=[f'feature_{i}' for i in range(X_train_bio.shape[1])])
train_df_bio['target'] = y_train_bio
test_df_bio = pd.DataFrame(X_test_bio, columns=[f'feature_{i}' for i in range(X_test_bio.shape[1])])

print(f"Standard Data - Train: {X_train_std.shape}, Test: {X_test_std.shape}")
print(f"Biological Data - Train: {X_train_bio.shape}, Test: {X_test_bio.shape}")

Using device: cpu
Standard Data - Train: (700, 20), Test: (300, 20)
Biological Data - Train: (350, 1000), Test: (150, 1000)


In [31]:
def train_eval_pytorch_tabular(model_config, train_df, test_df, y_test):
    # Configure Data
    data_config = DataConfig(target=['target'], continuous_cols=[col for col in train_df.columns if col != 'target'], categorical_cols=[])
    
    # Hide the model architecture summary table
    trainer_config = TrainerConfig(batch_size=64, max_epochs=30, progress_bar='none', trainer_kwargs={'enable_model_summary': False})

    # Initialize model with verbose=False to silence training logs
    model = TabularModel(
        data_config=data_config, 
        model_config=model_config, 
        optimizer_config=OptimizerConfig(), 
        trainer_config=trainer_config,
        verbose=False
    )

    model.fit(train_df)

    # Evaluate
    preds = model.predict(test_df)
    return accuracy_score(y_test, preds['target_prediction'])

# --- Execution ---
print("\n--- Training MLP ---")
mlp_config = CategoryEmbeddingModelConfig(task="classification", layers="512-256-128", dropout=0.4, activation="ReLU")

mlp_std_acc = train_eval_pytorch_tabular(mlp_config, train_df_std, test_df_std, y_test_std)
print(f"MLP Standard Accuracy: {mlp_std_acc:.3f}")

mlp_bio_acc = train_eval_pytorch_tabular(mlp_config, train_df_bio, test_df_bio, y_test_bio)
print(f"MLP Biological Accuracy: {mlp_bio_acc:.3f}")

Seed set to 42



--- Training MLP ---
MLP Standard Accuracy: 0.900


Seed set to 42


MLP Biological Accuracy: 0.547
